<!-- SPDX-License-Identifier: AGPL-3.0-or-later -->
<!-- Commercial license available -->
<!-- Copyright 1998-2026 Miroslav Sotek. All rights reserved. -->

# Balanced Resonate-and-Fire Evidence Notebook

This notebook verifies the SC-NeuroCore Balanced Resonate-and-Fire (BRF) implementation against the implemented Higuchi et al. 2024 Algorithm 1 update and the committed benchmark artifact.

## Evidence Boundary

This notebook proves only the local scalar BRF implementation and its committed evidence hooks:

- the divergence boundary formula `p(omega)`;
- exact one-step update order for `b_t`, `u_t`, `theta_t`, spike, and `q_t`;
- deterministic finite traces under fixed current sequences;
- fail-fast parameter validation for non-real boundary settings;
- presence and schema of the side-by-side benchmark artifact.

It does not claim reproduction of the full BRF-RSNN ICML training experiments, BHRF coverage, or hardware timing.

In [ ]:
from __future__ import annotations

import json
import math
import sys
from pathlib import Path
from typing import Any

import numpy as np

REPO = Path.cwd()
if not (REPO / "src" / "sc_neurocore").exists():
    REPO = REPO.parent
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

from sc_neurocore.neurons.models.balanced_resonate_and_fire import (
    BalancedResonateAndFireNeuron,
    sustain_oscillation_boundary,
)

BENCHMARK = REPO / "benchmarks/results/bench_balanced_resonate_and_fire.json"
print(f"Repository root: {REPO}")
print(f"Benchmark artifact: {BENCHMARK.relative_to(REPO)}")

## 1. Divergence Boundary

The implementation uses the real-valued discrete-time boundary

`p(omega) = (-1 + sqrt(1 - (dt * omega)^2)) / dt`.

The model rejects `dt * omega > 1` because the square-root term is no longer real.

In [ ]:
omega = 10.0
dt = 0.01
expected_boundary = (-1.0 + math.sqrt(1.0 - (dt * omega) ** 2)) / dt
actual_boundary = sustain_oscillation_boundary(omega, dt)

assert actual_boundary == actual_boundary
assert abs(actual_boundary - expected_boundary) < 1e-15

try:
    sustain_oscillation_boundary(omega=200.0, dt=0.01)
except ValueError as exc:
    invalid_boundary_message = str(exc)
else:
    raise AssertionError("dt * omega > 1 must fail")

print("p(omega):", actual_boundary)
print("invalid boundary guard:", invalid_boundary_message)

## 2. Exact One-Step Algorithm Check

This cell reproduces the scalar update manually and compares every state field after one model step.

In [ ]:
neuron = BalancedResonateAndFireNeuron(
    x=0.2,
    y=-0.1,
    q=0.3,
    omega=12.0,
    b_offset=0.75,
    threshold=1.0,
    gamma=0.9,
    dt=0.01,
)
current = 2.0

p_omega = sustain_oscillation_boundary(12.0, 0.01)
b_t = p_omega - 0.75 - 0.3
theta_t = 1.0 + 0.3
expected_x = 0.2 + 0.01 * (b_t * 0.2 - 12.0 * -0.1 + current)
expected_y = -0.1 + 0.01 * (12.0 * 0.2 + b_t * -0.1)
expected_spike = int(expected_x >= theta_t)
expected_q = 0.9 * 0.3 + expected_spike

spike = neuron.step(current)

assert spike == expected_spike
assert abs(neuron.x - expected_x) < 1e-15
assert abs(neuron.y - expected_y) < 1e-15
assert abs(neuron.q - expected_q) < 1e-15

print({"spike": spike, "x": neuron.x, "y": neuron.y, "q": neuron.q})

## 3. Deterministic Trace and Refractory Dynamics

The scalar model is deterministic. The refractory state raises the threshold after spikes and decays by `gamma` when no new spike occurs.

In [ ]:
left = BalancedResonateAndFireNeuron(omega=15.0, b_offset=1.5)
right = BalancedResonateAndFireNeuron(omega=15.0, b_offset=1.5)
currents = np.sin(np.linspace(0.0, 12.0, 200)) * 10.0 + 10.0

left_spikes = [left.step(float(current)) for current in currents]
right_spikes = [right.step(float(current)) for current in currents]

assert left_spikes == right_spikes
assert left.state() == right.state()
assert all(math.isfinite(value) for value in left.state().values())

refractory = BalancedResonateAndFireNeuron()
first_spike = refractory.step(200.0)
threshold_after_spike = refractory.dynamic_threshold
refractory.step(0.0)
threshold_after_decay = refractory.dynamic_threshold

assert first_spike == 1
assert threshold_after_spike == 2.0
assert 1.0 < threshold_after_decay < threshold_after_spike

print("trace spikes:", sum(left_spikes))
print("final state:", left.state())
print("thresholds:", {"after_spike": threshold_after_spike, "after_decay": threshold_after_decay})

## 4. Benchmark Artifact Check

The notebook reads the committed benchmark JSON and reports the side-by-side backend rows. It does not rerun benchmarks or replace the artifact.

In [ ]:
benchmark = json.loads(BENCHMARK.read_text(encoding="utf-8"))
assert benchmark["module"] == "sc_neurocore.neurons.models.balanced_resonate_and_fire"
assert "Higuchi" in benchmark["publication_reference"]

comparison_rows = benchmark["comparison"]
side_by_side = next(row for row in comparison_rows if row["workload"] == "scalar_200k_i2_omega10" and "python_step_ns" in row)
required_backends = ["python", "rust_pyo3", "go", "julia", "mojo"]
for backend in required_backends:
    assert side_by_side["backend_status"][backend] == "executed"

benchmark_table = {
    "python_step_ns": side_by_side["python_step_ns"],
    "rust_pyo3_step_ns": side_by_side["rust_pyo3_step_ns"],
    "go_step_ns": side_by_side["go_step_ns"],
    "julia_step_ns": side_by_side["julia_step_ns"],
    "mojo_step_ns": side_by_side["mojo_step_ns"],
}

print(json.dumps(benchmark_table, indent=2))

## 5. Evidence Manifest

This manifest ties the notebook run to local equations, deterministic trace checks, and the committed benchmark artifact.

In [ ]:
manifest = {
    "schema_version": "sc-neurocore.brf-evidence.v1",
    "reference": "Higuchi, Kairat, Bohte, and Otte (2024), Balanced Resonate-and-Fire Neurons",
    "boundary": {
        "p_omega": actual_boundary,
        "invalid_boundary_guard": "dt * omega > 1 raises ValueError",
    },
    "one_step": {
        "spike": spike,
        "x": neuron.x,
        "y": neuron.y,
        "q": neuron.q,
    },
    "trace": {
        "steps": len(currents),
        "spikes": int(sum(left_spikes)),
        "final_state": left.state(),
    },
    "benchmark_artifact": {
        "path": str(BENCHMARK.relative_to(REPO)),
        "schema_version": benchmark["schema_version"],
        "step_ns": benchmark_table,
    },
    "claim_boundary": "Scalar Algorithm 1 implementation and committed benchmark evidence only; no BRF-RSNN training reproduction or hardware timing claim.",
}

assert manifest["schema_version"] == "sc-neurocore.brf-evidence.v1"
assert manifest["benchmark_artifact"]["schema_version"] == 1
assert manifest["claim_boundary"].startswith("Scalar Algorithm 1")

print(json.dumps(manifest, indent=2))